# Archived research notebook

Original filename: `preprocessing sixth 步骤完备（必要）卡尔曼滤波那里用了cpu加速（弃）.ipynb`  
Purpose: Historic complete preprocessing variant with Kalman filtering.  
Historical research version retained for traceability. The original test data are unavailable; legacy outputs, execution state, and attachments were removed for public release.


In [ ]:
import mne
import os
import numpy as np
from pykalman import KalmanFilter
from mne.preprocessing import ICA
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

# 文件夹路径和参数设置
folder_path = r"data/chb-mit/chb01"
target_sfreq = 128  # 目标采样率
lowpass_freq = 80  # 低通滤波截止频率

def apply_kalman_filter(data, R=0.01):
    """卡尔曼滤波基线校正"""
    kf = KalmanFilter(
        initial_state_mean=0,
        initial_state_covariance=1,
        transition_matrices=1,
        observation_matrices=1,
        observation_covariance=R,
        transition_covariance=0.001
    )
    state_means, _ = kf.filter(data)
    return data - state_means.flatten()

def apply_zscore(raw_obj):
    """对连续数据（Raw对象）进行Z-分数归一化（按通道）"""
    data = raw_obj.get_data()
    mean = np.mean(data, axis=1, keepdims=True)  # 按通道计算均值
    std = np.std(data, axis=1, keepdims=True)     # 按通道计算标准差
    std[std == 0] = 1.0
    normalized_data = (data - mean) / std
    raw_obj._data = normalized_data
    return raw_obj

# 辅助函数：计算ICA成分的峰峰值
def compute_peak_to_peak(component):
    return np.max(component) - np.min(component)

# 辅助函数：计算ICA成分的功率谱密度
def compute_psd(component, sfreq):
    n = len(component)
    fft_result = np.fft.fft(component)
    psd = np.abs(fft_result) ** 2 / n
    freqs = np.fft.fftfreq(n, 1 / sfreq)
    return freqs[:n // 2], psd[:n // 2]

# 辅助函数：判断心电伪迹
def is_ecg_like(psd, freqs):
    powerline_ratio = np.sum(psd[(freqs >= 45) & (freqs <= 65)]) / np.sum(psd)
    return powerline_ratio > 0.1  # 可根据实际情况调整阈值

# 遍历文件夹中的processed.fif文件
for file in os.listdir(folder_path):
    if file.endswith("_processed.fif"):
        file_path = os.path.join(folder_path, file)
        try:
            # 1. 读取原始数据
            raw = mne.io.read_raw_fif(file_path, preload=True)
            print(f"处理文件: {file}")

            # 2. 低通滤波
            raw_filtered = raw.copy().filter(l_freq=None, h_freq=lowpass_freq, method="fir")

            # 3. 下采样
            raw_downsampled = raw_filtered.copy().resample(
                sfreq=target_sfreq, method="polyphase", n_jobs=-1, verbose=False
            )
            print(f"完成下采样至 {target_sfreq} Hz")

            # 4. 使用卡尔曼滤波进行基线校正
            data = raw_downsampled.get_data()
            num_channels = data.shape[0]
            processed_channels = Parallel(n_jobs=-1)(
                delayed(apply_kalman_filter)(data[i, :]) for i in range(num_channels)
            )
            data = np.array(processed_channels)
            raw_downsampled._data = data
            print("完成卡尔曼滤波基线校正")

            # 5. 使用Picard算法进行ICA（在连续数据上）
            n_components = 0.99  # 保留99%的方差
            ica = ICA(
                n_components=n_components,
                method='picard',
                random_state=42,
                max_iter=500
            )
            ica.fit(raw_downsampled)
            print("完成ICA拟合")

            # 分析ICA成分，识别更多伪迹成分
            components = ica.get_sources(raw_downsampled).get_data()
            peak_to_peak_values = [compute_peak_to_peak(comp) for comp in components]
            sfreq = raw_downsampled.info['sfreq']
            for i, component in enumerate(components):
                freqs, psd = compute_psd(component, sfreq)
                # 眼电伪迹判断
                eog_like = np.sum(psd[(freqs >= 0) & (freqs < 5)]) / np.sum(psd) > 0.5
                # 肌电伪迹判断
                emg_like = np.sum(psd[(freqs >= 50)]) / np.sum(psd) > 0.3
                # 心电伪迹判断
                ecg_like = is_ecg_like(psd, freqs)
                if eog_like or emg_like or ecg_like or peak_to_peak_values[i] > np.percentile(peak_to_peak_values, 95):
                    ica.exclude.append(i)

            print(f"检测到 {len(ica.exclude)} 个伪迹相关ICA成分")

            # 应用ICA校正到连续数据
            raw_cleaned = ica.apply(raw_downsampled.copy())
            print("完成ICA伪迹移除")

            # **关键修正：对连续数据进行Z-分数归一化（按通道）**
            raw_normalized = apply_zscore(raw_cleaned)
            print("完成Z-分数归一化")

            # 保存结果（直接保存连续数据，无分段）
            output_file = file_path.replace("_processed.fif", f"_processed_full_{target_sfreq}Hz.fif")
            raw_normalized.save(output_file, overwrite=True)
            print(f"保存至: {output_file}，采样率: {target_sfreq} Hz")

        except Exception as e:
            print(f"处理失败: {file}，错误信息: {e}")
            continue